# ChiFraud 簡繁雙模型實驗（Kaggle）

此 notebook 只負責固定來源並呼叫 repo 內的正式訓練流程。2022 與 2023 都會進入 train／validation／test；test 不參與 checkpoint、溫度或門檻選擇。資料不重新發布，直接從官方 ChiFraud commit 取得。

In [ ]:
from pathlib import Path
import os, subprocess, sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('請把 working directory 指到 AI_Voice repo 根目錄。')
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)


In [ ]:
CHIFRAUD_REVISION = '5a0245743e85154f114f2b3ea24b0bed4e64cf0c'
BERT_REVISION = '8f23c25b06e129b6c986331a13d8d025a92cf0ea'
source_dir = Path('/kaggle/working/ChiFraud')
if not source_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/xuemingxxx/ChiFraud.git', str(source_dir)], check=True)
subprocess.run(['git', '-C', str(source_dir), 'checkout', '--detach', CHIFRAUD_REVISION], check=True)
subprocess.run([sys.executable, '-m', 'src.chifraud_data', str(source_dir / 'dataset'), '/kaggle/working/chifraud_prepared', '--source-revision', CHIFRAUD_REVISION, '--seed', '42'], check=True)


In [ ]:
subprocess.run([
    sys.executable, '-m', 'scripts.run_chifraud_experiment',
    '/kaggle/working/chifraud_prepared', '/kaggle/working/chifraud_experiment',
    '--base-model', 'google-bert/bert-base-chinese',
    '--base-revision', BERT_REVISION,
    '--seeds', '42',
    '--max-epochs', '8', '--patience', '2', '--batch-size', '16'
], check=True)


In [ ]:
import json
report_path = Path('/kaggle/working/chifraud_experiment/selection_report.json')
report = json.loads(report_path.read_text(encoding='utf-8'))
print(json.dumps(report['selection'], ensure_ascii=False, indent=2))
print('若 status=additional_seeds_required，請在全新 output 以 --seeds 42 43 44 重跑；只有 status=selected 且候選通過驗收才可下載升級。')
